In [41]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score, balanced_accuracy_score,
    confusion_matrix
)
from itertools import product, pairwise
from tqdm.notebook import tqdm
from typing import List, Tuple, Optional

from data_utils.load_datasets import load_appl_ind
from comp_modeling.compxconstraints import CompXConstraints

In [2]:
rng = np.random.default_rng(579372518)

In [3]:
appl_ind_path = "resources/developed/merged_geco_sp_appl_ind.tsv"
appl_ind = load_appl_ind(
	appl_ind_path=appl_ind_path,
    # this dataset is clean
	remove_na_columns=False,
	remove_unconfirmed=False,
    remove_missing=False
)
appl_ind.sample(3)

,linker,def-0_is_applicable,def-0_applies,p2l:decl_cl:pl:#0-0|def-0_is_applicable,p2l:decl_cl:pl:#0-0|def-0_applies,p2l:decl_cl:pl:#s-0_is_applicable,p2l:decl_cl:pl:#s-0_applies,p2l:decl_cl:pl:#e-0|def-0_is_applicable,p2l:decl_cl:pl:#e-0|def-0_applies,p2l:decl_cl:pl:#e-0|msyl_anim-e_is_applicable,...,p2l:drv:deverb_schwa$-0|pl_interpr-en_is_applicable,p2l:drv:sfx:sfx-s|pl_interpr-en_is_applicable,p2l:drv:sfx:sfx-s|pl_interpr-en_applies,p2l:phon_fin:F_#stressed_phon$|pl_interpr-en_is_applicable,p2l:phon_fin:weak_F_cons$-0/s/en|!pl_interpr-0/s_is_applicable,p2l:phon_fin:weak_F_cons$-0/s/en|!pl_interpr-0/s_applies,p2l:phon_fin:weak_F_cons$-0/s/en|pl_interpr-en_is_applicable,p2l:phon_fin:weak_F_cons$-0/s/en|pl_interpr-en_applies,l2p:pl_marker|def-pl_is_applicable,l2p:pl_marker|def-pl_applies
comp_gecodb,,,,,,,,,,,,,,,,,,,,,
zahn_operation,_,True,True,False,NaN,False,NaN,False,NaN,False,...,False,False,NaN,False,False,NaN,False,NaN,False,NaN
kind_+er_krankheit,_+=er_,True,False,False,NaN,False,NaN,False,NaN,False,...,False,False,NaN,False,False,NaN,False,NaN,True,False
kind_+er_reim,_+=er_,True,False,False,NaN,False,NaN,False,NaN,False,...,False,False,NaN,False,False,NaN,False,NaN,True,False


### Short Demonstration of `CompXConstraints`

##### 'Kinderwagen'

In [4]:
comp_gecodb = "kind_+er_wagen"
appl_row = appl_ind.loc[comp_gecodb]
cxc = CompXConstraints(appl_row)
cxc.true

'_+=er_'

In [5]:
cxc.acceptable_linkers

defaultdict(list,
            {'_': ['def-0',
              'p2l:decl_cl:pl:#er-0/er|def-0/er',
              'p2l:decl_cl:pl:#er-0/er|!pl_interpr-0'],
             '_+=er_': ['p2l:decl_cl:pl:#er-0/er|def-0/er',
              'p2l:decl_cl:pl:#er-0/er|anim-er']})

In [6]:
cxc.unacceptable_linkers

defaultdict(list,
            {'_+s_': ['l2p:s|tend_cmpx_morph', 'l2p:s|tend_cmpx_phon'],
             '_+en_': ['l2p:en|par'],
             '_+e_': ['l2p:e|par'],
             '_+=e_': ['l2p:e_uml'],
             '_+=_': ['l2p:0_uml']})

In [7]:
cxc.predict(
    method="albright_hayes",
    count_def_constr=False,
    constr_cooperation=False,
    restrict_with_l2p=False
)

cxc.gen_constr_df

,linker,condition,spec_score,cvg,reg,confidence
c_id,,,,,,
p2l:decl_cl:pl:#er-0/er|!pl_interpr-0,_,[PL:er] & [sem:!PL],25.2,0.171,0.622,0.6220
p2l:decl_cl:pl:#er-0/er|anim-er,_+=er_,[PL:er] & [[sem:anim] | [sem:pers]],28.0,0.003,0.501,0.5010
p2l:decl_cl:pl:#er-0/er|def-0/er,_,[PL:er],14.0,0.021,0.815,0.4075
p2l:decl_cl:pl:#er-0/er|def-0/er,_+=er_,[PL:er],14.0,0.021,0.815,0.4075


In [8]:
cxc.linker_scoreboard

,confidence,cvg,c_id
linker,,,
_,0.622,0.171,p2l:decl_cl:pl:#er-0/er|!pl_interpr-0
_+=er_,0.501,0.003,p2l:decl_cl:pl:#er-0/er|anim-er


In [9]:
cxc.y_pred

'_'

In [10]:
cxc.predict(
    method="albright_hayes",
    count_def_constr=True,
    constr_cooperation=True,
    restrict_with_l2p=False
)
 
cxc.linker_scoreboard

,confidence,cvg,c_id
linker,,,
_,1.0295,0.192,"p2l:decl_cl:pl:#er-0/er|def-0/er, p2l:decl_cl:..."
_+=er_,0.9085,0.024,"p2l:decl_cl:pl:#er-0/er|def-0/er, p2l:decl_cl:..."


##### 'Geschenkeliste'

In [11]:
comp_gecodb = "geschenk_+e_liste"
appl_row = appl_ind.loc[comp_gecodb]
cxc = CompXConstraints(appl_row)
cxc.true

'_+e_'

In [12]:
cxc.predict(
    method="albright_hayes",
    count_def_constr=True,
    constr_cooperation=False,
    restrict_with_l2p=True
)

cxc.restr_constr_df

,linker,condition,spec_score,cvg,reg,confidence
c_id,,,,,,
l2p:er|par,_+=er_,[PL:er],14.0,0.009,1.000,1.000
l2p:e_uml,_+=e_,[PL:ue],14.0,0.003,1.000,1.000
l2p:0_uml,_+=_,[PL:uo],14.0,0.001,1.000,1.000
l2p:en|par,_+en_,[PL:en],14.0,0.152,0.985,0.985
l2p:es|mono_syl,_+es_,[nsyl:mono] | [[nsyl:mono] & [drv:PRX]],20.4,0.010,0.972,0.972
l2p:er|smpx,_+=er_,[drv:smpx],14.0,0.009,0.903,0.903
l2p:e|smpx,_+e_,[drv:smpx],14.0,0.009,0.806,0.806


In [13]:
cxc.linker_scoreboard

,confidence,cvg,c_id,removed_by
linker,,,,
_,0.754,0.162,p2l:decl_cl:pl:#e-0|def-0,
_+s_,0.649,0.159,p2l:drv:prx_deverb-s,
_+e_,0.544,0.079,p2l:decl_cl:pl:#e-0|pl_interpr-e,l2p:e|smpx


In [14]:
cxc.predict(
    method="albright_hayes",
    count_def_constr=False,
    constr_cooperation=True,
    restrict_with_l2p=True
)

cxc.linker_scoreboard

,confidence,cvg,c_id,removed_by
linker,,,,
_,0.754,0.162,p2l:decl_cl:pl:#e-0|def-0,
_+s_,0.649,0.159,p2l:drv:prx_deverb-s,
_+e_,0.544,0.079,p2l:decl_cl:pl:#e-0|pl_interpr-e,l2p:e|smpx


In [15]:
cxc.predict(
    method="yang-cond",
    count_def_constr=True,
    constr_cooperation=False,
    restrict_with_l2p=True
)

cxc.linker_scoreboard

,confidence,cvg,c_id,removed_by
linker,,,,
_+e_,26.8,0.079,p2l:decl_cl:pl:#e-0|pl_interpr-e,
_,14.0,0.162,p2l:decl_cl:pl:#e-0|def-0,
_+s_,12.7,0.159,p2l:drv:prx_deverb-s,


In [16]:
cxc.predict(
    method="yang-cvg",
    count_def_constr=True,
    constr_cooperation=False,
    restrict_with_l2p=True
)

cxc.linker_scoreboard

,confidence,reg,c_id,removed_by
linker,,,,
_+e_,0.079,0.544,p2l:decl_cl:pl:#e-0|pl_interpr-e,
_+s_,0.159,0.649,p2l:drv:prx_deverb-s,
_,0.162,0.754,p2l:decl_cl:pl:#e-0|def-0,


### Prerequisites

In [17]:
N = len(appl_ind)

cXc = [
    CompXConstraints(row)
    for _, row in tqdm(
        appl_ind.iterrows(),
        total=N
    )
]

Y_true = appl_ind["linker"].to_list()

  0%|          | 0/882 [00:00<?, ?it/s]

In [18]:
def calculate_ldistr(linkers: List[str]) -> pd.Series:
    n = len(linkers)
    unique_linkers = np.unique(linkers)
    return pd.Series({
        l: linkers.count(l) / n
        for l in unique_linkers
    }).sort_values(ascending=False)#.round(3)

In [19]:
ldistr = calculate_ldistr(Y_true)
ldistr.round(3)

_         0.570
_+en_     0.186
_+=er_    0.160
_+e_      0.048
_+=e_     0.015
_+s_      0.015
_+=_      0.007
dtype: float64

In [20]:
def get_eval_df(Y_pred: List[str]) -> pd.DataFrame:
    return pd.DataFrame({
        "true": Y_true,
        "pred": Y_pred
    })

In [21]:
def compare_ldistr(linkers: List[str]) -> pd.DataFrame:
    test_ldistr = calculate_ldistr(linkers).to_frame("ours")
    return ldistr.to_frame(name="reference")    \
        .join(test_ldistr)  \
        .fillna(0)  \
        .round(3)

In [22]:
def calculate_accuracies(eval_df: pd.DataFrame) -> pd.Series:
    accuracies_data = {}
    for linker in ldistr.index:
        eval_df_linker = eval_df[eval_df["true"] == linker]
        acc = accuracy_score(eval_df_linker["true"], eval_df_linker["pred"])
        accuracies_data[linker] = acc
    accuracies_df = pd.Series(accuracies_data, index=ldistr.index)
    overall_acc = accuracy_score(eval_df["true"], eval_df["pred"])
    balanced_acc = balanced_accuracy_score(eval_df["true"], eval_df["pred"])
    accuracies_df = pd.concat((
        pd.Series({
            "overall": overall_acc,
            "balanced": balanced_acc
        }),
        accuracies_df
    ))
    return accuracies_df.round(3)

In [23]:
def calculate_f1s(Y_pred: List[str]) -> pd.DataFrame:
    f1_df = pd.DataFrame(
        precision_recall_fscore_support(
            Y_true, Y_pred,
            labels=ldistr.index,
            zero_division=0
        ),
        index=["prec", "recall", "f1", "N"],
        columns=ldistr.index
    )
    f1_df.insert(0, "avg", f1_df.mean(axis=1))
    f1_df = f1_df.T
    return f1_df.round(3)

In [24]:
def calculate_confusion_matrix(Y_pred: List[str]) -> pd.DataFrame:
    return pd.DataFrame(
        confusion_matrix(
            Y_true, Y_pred,
            labels=ldistr.index,
            normalize="true"
        ),
        index=ldistr.index,
        columns=ldistr.index,
    ).round(3)

In [25]:
def calculate_metrics(
    Y_pred: List[str]
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.DataFrame]:
    eval_df = get_eval_df(Y_pred)
    ldistr_comparison = compare_ldistr(Y_pred)
    accuracies_df = calculate_accuracies(eval_df)
    f1_df = calculate_f1s(Y_pred)
    confusion_matrix = calculate_confusion_matrix(Y_pred)
    return ldistr_comparison, accuracies_df, f1_df, confusion_matrix

## Modeling

### Baseline 1: Zero Linker Only

In [26]:
Y_pred_zero = ["_"] * N
_, accuracies_df, f1_df, cnf_matrix = calculate_metrics(Y_pred_zero)

In [27]:
accuracies_df

overall     0.570
balanced    0.143
_           1.000
_+en_       0.000
_+=er_      0.000
_+e_        0.000
_+=e_       0.000
_+s_        0.000
_+=_        0.000
dtype: float64

In [28]:
f1_df

,prec,recall,f1,N
avg,0.081,0.143,0.104,126.0
_,0.570,1.000,0.726,503.0
_+en_,0.000,0.000,0.000,164.0
_+=er_,0.000,0.000,0.000,141.0
_+e_,0.000,0.000,0.000,42.0
_+=e_,0.000,0.000,0.000,13.0
_+s_,0.000,0.000,0.000,13.0
_+=_,0.000,0.000,0.000,6.0


In [29]:
cnf_matrix

,_,_+en_,_+=er_,_+e_,_+=e_,_+s_,_+=_
_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+en_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+=er_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+e_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+=e_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+s_,1.0,0.0,0.0,0.0,0.0,0.0,0.0
_+=_,1.0,0.0,0.0,0.0,0.0,0.0,0.0


### Baseline 2: Random Distribution-Aware Sampling

In [30]:
Y_pred_rand = rng.choice(ldistr.index, size=N, replace=True, p=ldistr)
Y_pred_rand = Y_pred_rand.tolist()
ldistr_comparison, accuracies_df, f1_df, cnf_matrix = calculate_metrics(Y_pred_rand)

In [31]:
ldistr_comparison

,reference,ours
_,0.570,0.567
_+en_,0.186,0.170
_+=er_,0.160,0.176
_+e_,0.048,0.048
_+=e_,0.015,0.017
_+s_,0.015,0.014
_+=_,0.007,0.009


In [32]:
accuracies_df

overall     0.375
balanced    0.127
_           0.563
_+en_       0.171
_+=er_      0.135
_+e_        0.024
_+=e_       0.000
_+s_        0.000
_+=_        0.000
dtype: float64

In [33]:
f1_df

,prec,recall,f1,N
avg,0.128,0.127,0.128,126.0
_,0.566,0.563,0.564,503.0
_+en_,0.187,0.171,0.178,164.0
_+=er_,0.123,0.135,0.128,141.0
_+e_,0.024,0.024,0.024,42.0
_+=e_,0.000,0.000,0.000,13.0
_+s_,0.000,0.000,0.000,13.0
_+=_,0.000,0.000,0.000,6.0


In [34]:
cnf_matrix

,_,_+en_,_+=er_,_+e_,_+=e_,_+s_,_+=_
_,0.563,0.155,0.197,0.050,0.010,0.014,0.012
_+en_,0.555,0.171,0.177,0.055,0.037,0.006,0.000
_+=er_,0.553,0.227,0.135,0.043,0.028,0.007,0.007
_+e_,0.619,0.143,0.119,0.024,0.000,0.071,0.024
_+=e_,0.923,0.000,0.077,0.000,0.000,0.000,0.000
_+s_,0.538,0.231,0.154,0.077,0.000,0.000,0.000
_+=_,0.500,0.500,0.000,0.000,0.000,0.000,0.000


### Albright & Hayes 2002

In [35]:
def predict_ah02(
    count_def_constr: bool,
    constr_cooperation: bool,
    restrict_with_l2p: bool,
    verbose: Optional[bool]=True,
    tqdm_prx: Optional[str]=""
) -> List[str]:
    _cXc = tqdm(cXc, desc=tqdm_prx) if verbose else cXc
    return [
        cxc.predict(
            method="albright_hayes",
            count_def_constr=count_def_constr,
            constr_cooperation=constr_cooperation,
            restrict_with_l2p=restrict_with_l2p
        )
        for cxc in _cXc
    ]

In [36]:
Y_pred_ah02 = {}
ah02_metrics = {}
s = lambda x: "+" if x else "-"
for count_def, coop, l2p in product([True, False], repeat=3):
    key = (
        f"{s(count_def)}count_def|"
        f"{s(coop)}coop|"
        f"{s(l2p)}l2p"
    )
    Y_pred_ah02[key] = [
        cxc.predict(
            method="albright_hayes",
            count_def_constr=count_def,
            constr_cooperation=coop,
            restrict_with_l2p=l2p
        )
        for cxc in tqdm(cXc)
    ]
    ah02_metrics[key] = calculate_metrics(Y_pred_ah02[key])

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

In [ ]:
# * Three worst results are with -l2p
# * Cooperation does not improve the scores
# * The low metrics are caused by vistually absent
# possibility to produce pluralic linkers due to
# the low regularity of the semantic constraints
metrics_ah02_df = pd.DataFrame({
    key: {
        "acc": metrics[1].loc["overall"],
        "acc_balanced": metrics[1].loc["balanced"],
        "f1_avg": metrics[2].loc["avg", "f1"]
    }
    for key, metrics in ah02_metrics.items()
}).T.sort_values(by=["f1_avg", "acc_balanced", "acc"], ascending=False)

metrics_ah02_df

,acc,acc_balanced,f1_avg
+count_def|+coop|+l2p,0.667,0.224,0.230
+count_def|+coop|-l2p,0.667,0.224,0.230
+count_def|-coop|+l2p,0.667,0.224,0.230
-count_def|+coop|+l2p,0.667,0.224,0.230
-count_def|-coop|+l2p,0.667,0.224,0.230
-count_def|+coop|-l2p,0.667,0.224,0.223
+count_def|-coop|-l2p,0.660,0.223,0.221
-count_def|-coop|-l2p,0.660,0.223,0.221


In [38]:
ldistr_comparison, accuracies_df, f1_df, cnf_matrix = ah02_metrics["+count_def|+coop|+l2p"]

In [39]:
ldistr_comparison

,reference,ours
_,0.570,0.882
_+en_,0.186,0.090
_+=er_,0.160,0.027
_+e_,0.048,0.000
_+=e_,0.015,0.000
_+s_,0.015,0.001
_+=_,0.007,0.000


In [40]:
cnf_matrix

,_,_+en_,_+=er_,_+e_,_+=e_,_+s_,_+=_
_,0.982,0.002,0.016,0.0,0.0,0.000,0.0
_+en_,0.518,0.476,0.000,0.0,0.0,0.006,0.0
_+=er_,0.887,0.000,0.113,0.0,0.0,0.000,0.0
_+e_,1.000,0.000,0.000,0.0,0.0,0.000,0.0
_+=e_,1.000,0.000,0.000,0.0,0.0,0.000,0.0
_+s_,1.000,0.000,0.000,0.0,0.0,0.000,0.0
_+=_,1.000,0.000,0.000,0.0,0.0,0.000,0.0


In [49]:
for (key_1, preds_1), (key_2, preds_2) in pairwise(Y_pred_ah02.items()):
    print(key_1, "\t", key_2)
    print(round(accuracy_score(preds_1, preds_2), 3))
    print()

+count_def|+coop|+l2p 	 +count_def|+coop|-l2p
1.0

+count_def|+coop|-l2p 	 +count_def|-coop|+l2p
1.0

+count_def|-coop|+l2p 	 +count_def|-coop|-l2p
0.965

+count_def|-coop|-l2p 	 -count_def|+coop|+l2p
0.965

-count_def|+coop|+l2p 	 -count_def|+coop|-l2p
0.972

-count_def|+coop|-l2p 	 -count_def|-coop|+l2p
0.972

-count_def|-coop|+l2p 	 -count_def|-coop|-l2p
0.965



In [50]:
Y_pred_y16_cond = {}
y16_cond_metrics = {}
s = lambda x: "+" if x else "-"
for count_def, coop, l2p in product([True, False], repeat=3):
    key = (
        f"{s(count_def)}count_def|"
        f"{s(coop)}coop|"
        f"{s(l2p)}l2p"
    )
    Y_pred_y16_cond[key] = [
        cxc.predict(
            method="yang-cond",
            count_def_constr=count_def,
            constr_cooperation=coop,
            restrict_with_l2p=l2p
        )
        for cxc in tqdm(cXc)
    ]
    y16_cond_metrics[key] = calculate_metrics(Y_pred_y16_cond[key])

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

In [ ]:
# * All the results are identical
# * L2P cannot restrict anything because their
# conditions are always quite general
metrics_y16_cond_df = pd.DataFrame({
    key: {
        "acc": metrics[1].loc["overall"],
        "acc_balanced": metrics[1].loc["balanced"],
        "f1_avg": metrics[2].loc["avg", "f1"]
    }
    for key, metrics in y16_cond_metrics.items()
}).T.sort_values(by=["f1_avg", "acc_balanced", "acc"], ascending=False)

metrics_y16_cond_df

,acc,acc_balanced,f1_avg
+count_def|+coop|+l2p,0.713,0.361,0.397
+count_def|+coop|-l2p,0.713,0.361,0.397
+count_def|-coop|+l2p,0.713,0.361,0.397
+count_def|-coop|-l2p,0.713,0.361,0.397
-count_def|+coop|+l2p,0.713,0.361,0.397
-count_def|+coop|-l2p,0.713,0.361,0.397
-count_def|-coop|+l2p,0.713,0.361,0.397
-count_def|-coop|-l2p,0.713,0.361,0.397


In [52]:
for (key_1, preds_1), (key_2, preds_2) in pairwise(Y_pred_y16_cond.items()):
    print(key_1, "\t", key_2)
    print(round(accuracy_score(preds_1, preds_2), 3))
    print()

+count_def|+coop|+l2p 	 +count_def|+coop|-l2p
1.0

+count_def|+coop|-l2p 	 +count_def|-coop|+l2p
0.999

+count_def|-coop|+l2p 	 +count_def|-coop|-l2p
1.0

+count_def|-coop|-l2p 	 -count_def|+coop|+l2p
0.999

-count_def|+coop|+l2p 	 -count_def|+coop|-l2p
1.0

-count_def|+coop|-l2p 	 -count_def|-coop|+l2p
0.999

-count_def|-coop|+l2p 	 -count_def|-coop|-l2p
1.0



In [53]:
ldistr_comparison, accuracies_df, f1_df, cnf_matrix = y16_cond_metrics["+count_def|+coop|+l2p"]

In [54]:
ldistr_comparison

,reference,ours
_,0.570,0.748
_+en_,0.186,0.125
_+=er_,0.160,0.101
_+e_,0.048,0.010
_+=e_,0.015,0.007
_+s_,0.015,0.001
_+=_,0.007,0.008


In [55]:
f1_df

,prec,recall,f1,N
avg,0.502,0.361,0.397,126.0
_,0.717,0.940,0.813,503.0
_+en_,0.709,0.476,0.569,164.0
_+=er_,0.775,0.489,0.600,141.0
_+e_,0.333,0.071,0.118,42.0
_+=e_,0.833,0.385,0.526,13.0
_+s_,0.000,0.000,0.000,13.0
_+=_,0.143,0.167,0.154,6.0


In [56]:
cnf_matrix

,_,_+en_,_+=er_,_+e_,_+=e_,_+s_,_+=_
_,0.940,0.014,0.020,0.012,0.002,0.000,0.012
_+en_,0.518,0.476,0.000,0.000,0.000,0.006,0.000
_+=er_,0.511,0.000,0.489,0.000,0.000,0.000,0.000
_+e_,0.381,0.548,0.000,0.071,0.000,0.000,0.000
_+=e_,0.615,0.000,0.000,0.000,0.385,0.000,0.000
_+s_,0.077,0.154,0.769,0.000,0.000,0.000,0.000
_+=_,0.833,0.000,0.000,0.000,0.000,0.000,0.167


In [65]:
Y_pred_y16_cvg = {}
y16_cvg_metrics = {}
s = lambda x: "+" if x else "-"
for count_def, coop, l2p in product([True, False], repeat=3):
    key = (
        f"{s(count_def)}count_def|"
        f"{s(coop)}coop|"
        f"{s(l2p)}l2p"
    )
    Y_pred_y16_cvg[key] = [
        cxc.predict(
            method="yang-cvg",
            count_def_constr=count_def,
            constr_cooperation=coop,
            restrict_with_l2p=l2p
        )
        for cxc in tqdm(cXc)
    ]
    y16_cvg_metrics[key] = calculate_metrics(Y_pred_y16_cvg[key])

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

  0%|          | 0/882 [00:00<?, ?it/s]

In [ ]:
# * All four runs with -L2P have the worst results:
# combined with a similar result from AH02, we conclude
# that L2P do play a role
# * Removing cooperation produces better results:
# combined with a similar result from AH02, we conclude
# that compounding constraints do not cooperate
# * On the other hand, relying on cvg only produces too many
# "rare" linkers where they shouldn't be
metrics_y16_cvg_df = pd.DataFrame({
    key: {
        "acc": metrics[1].loc["overall"],
        "acc_balanced": metrics[1].loc["balanced"],
        "f1_avg": metrics[2].loc["avg", "f1"]
    }
    for key, metrics in y16_cvg_metrics.items()
}).T.sort_values(by=["f1_avg", "acc_balanced", "acc"], ascending=False)

metrics_y16_cvg_df

,acc,acc_balanced,f1_avg
+count_def|-coop|+l2p,0.715,0.395,0.446
-count_def|-coop|+l2p,0.715,0.395,0.446
+count_def|+coop|+l2p,0.590,0.421,0.433
-count_def|+coop|+l2p,0.590,0.421,0.433
+count_def|-coop|-l2p,0.689,0.316,0.357
-count_def|-coop|-l2p,0.689,0.316,0.357
+count_def|+coop|-l2p,0.563,0.343,0.343
-count_def|+coop|-l2p,0.563,0.343,0.343


In [67]:
for (key_1, preds_1), (key_2, preds_2) in pairwise(Y_pred_y16_cvg.items()):
    print(key_1, "\t", key_2)
    print(round(accuracy_score(preds_1, preds_2), 3))
    print()

+count_def|+coop|+l2p 	 +count_def|+coop|-l2p
0.965

+count_def|+coop|-l2p 	 +count_def|-coop|+l2p
0.618

+count_def|-coop|+l2p 	 +count_def|-coop|-l2p
0.965

+count_def|-coop|-l2p 	 -count_def|+coop|+l2p
0.618

-count_def|+coop|+l2p 	 -count_def|+coop|-l2p
0.965

-count_def|+coop|-l2p 	 -count_def|-coop|+l2p
0.618

-count_def|-coop|+l2p 	 -count_def|-coop|-l2p
0.965



In [68]:
ldistr_comparison, accuracies_df, f1_df, cnf_matrix = y16_cvg_metrics["+count_def|+coop|+l2p"]

In [69]:
ldistr_comparison

,reference,ours
_,0.570,0.460
_+en_,0.186,0.104
_+=er_,0.160,0.370
_+e_,0.048,0.039
_+=e_,0.015,0.007
_+s_,0.015,0.020
_+=_,0.007,0.000


In [70]:
f1_df

,prec,recall,f1,N
avg,0.518,0.421,0.433,126.0
_,0.692,0.559,0.618,503.0
_+en_,0.967,0.543,0.695,164.0
_+=er_,0.365,0.844,0.510,141.0
_+e_,0.765,0.619,0.684,42.0
_+=e_,0.833,0.385,0.526,13.0
_+s_,0.000,0.000,0.000,13.0
_+=_,0.000,0.000,0.000,6.0


In [71]:
cnf_matrix

,_,_+en_,_+=er_,_+e_,_+=e_,_+s_,_+=_
_,0.559,0.006,0.392,0.012,0.002,0.030,0.0
_+en_,0.457,0.543,0.000,0.000,0.000,0.000,0.0
_+=er_,0.156,0.000,0.844,0.000,0.000,0.000,0.0
_+e_,0.310,0.000,0.000,0.619,0.000,0.071,0.0
_+=e_,0.615,0.000,0.000,0.000,0.385,0.000,0.0
_+s_,0.077,0.000,0.769,0.154,0.000,0.000,0.0
_+=_,1.000,0.000,0.000,0.000,0.000,0.000,0.0
